# 🔗 IntelliCode-SL | Formatter SLM — Merge Adapter into Base Model

**What this does:**  
Loads the base model and LoRA adapter both at full fp16 precision, merges them into a single unified model, and saves the merged model to Google Drive.

**Why merge?**  
- No adapter loading overhead at inference time  
- Single model file — simpler to load and deploy  
- Full fp16 precision throughout — no quantization  
- Slightly faster inference since LoRA math is baked in  

**Input:**  `MyDrive/IntelliCode-SL/adapters/formatter_adapter/`  
**Output:** `MyDrive/IntelliCode-SL/merged_models/formatter_merged/`

> T4 GPU runtime required. Run cells top to bottom.

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────
!pip install -q transformers peft accelerate
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────
import os, gc, torch
from google.colab import drive
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("✅ Imports done")
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

In [ ]:
# ── Cell 3: Config + Drive + Login ─────────────────────────────
drive.mount("/content/drive")

HF_TOKEN     = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX"  # ← paste your token
MODEL_NAME   = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/IntelliCode-SL/adapters/formatter_adapter"
SAVE_PATH    = "/content/drive/MyDrive/IntelliCode-SL/merged_models/formatter_merged"

login(token=HF_TOKEN)
os.makedirs(SAVE_PATH, exist_ok=True)

print("✅ Drive mounted + HF login done")
print(f"   Base model   : {MODEL_NAME}")
print(f"   Adapter path : {ADAPTER_PATH}")
print(f"   Save path    : {SAVE_PATH}")

In [ ]:
# ── Cell 4: Load Base Model at fp16 ───────────────────────────
# No quantization (load_in_4bit=False by default)
# torch.float16 keeps full 16-bit precision throughout
print("Loading base model at fp16 (no quantization)...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype = torch.float16,
    device_map  = "auto",
    token       = HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token = HF_TOKEN
)

print("✅ Base model loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 5: Load Adapter at fp16 ──────────────────────────────
# Loads the LoRA adapter saved during fine-tuning in notebook 04
print("Loading LoRA adapter at fp16...")

model_with_adapter = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    torch_dtype = torch.float16
)

print("✅ Adapter loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 6: Merge Adapter into Base Model ─────────────────────
# merge_and_unload() mathematically bakes the LoRA weights
# into the base model weights and returns a plain model
# with no adapter — same as the base model class, just
# with the fine-tuned weights incorporated.
print("Merging adapter into base model...")

merged_model = model_with_adapter.merge_and_unload()

print("✅ Merge complete")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"   Model type: {type(merged_model).__name__}")

In [ ]:
# ── Cell 7: Save Merged Model to Drive ────────────────────────
print(f"Saving merged model to Drive...")
print(f"   Path: {SAVE_PATH}")
print(f"   (This may take a few minutes)")

merged_model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Merged model saved")

# Show what was saved
files = os.listdir(SAVE_PATH)
print(f"\n   Files saved ({len(files)} total):")
for f in sorted(files):
    size_mb = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f"   {f:<40} {size_mb:>8.1f} MB")

In [ ]:
# ── Cell 8: Verify — Load Merged Model Fresh from Drive ────────
# Frees VRAM and reloads from the saved path to confirm
# the model was written correctly and is fully functional.
print("Verifying saved model by loading fresh from Drive...")

# Free existing models from VRAM
del merged_model, model_with_adapter, base_model
gc.collect()
torch.cuda.empty_cache()

# Load fresh from saved path
verify_model = AutoModelForCausalLM.from_pretrained(
    SAVE_PATH,
    torch_dtype = torch.float16,
    device_map  = "auto",
)
verify_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
verify_model.eval()

print("✅ Merged model loaded fresh from Drive")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 9: Quick Inference Test ──────────────────────────────
# Uses the same prompt template from fine-tuning (notebook 04)
# so the merged model sees the exact format it was trained on.
PROMPT_TEMPLATE = """### Task:
You are a response formatter. Take the raw agent output below and rewrite it as a clean, professional, well-structured response that is easy for the user to read and understand.
Do not add new information. Only improve the clarity, structure, and readability of the existing content.

### Raw Agent Output:
{}

### Formatted Response:
"""

test_cases = [
    # Debug output
    """- bug found on line 8: used + instead of -
- also missing null check at start
- fixed both issues
- function now returns correct result""",

    # Generate output
    """- wrote function is_prime(n)
- handles edge cases: n < 2 returns False
- checks divisibility up to sqrt(n)
- returns True if no divisors found""",

    # Explain output
    """- function takes list as input
- iterates through each element
- multiplies each by 2
- returns new list with doubled values""",

    # Modify output
    """- added type hints to all parameters
- added docstring explaining purpose
- added input validation for negative numbers
- refactored loop to list comprehension
- added logging for debug purposes""",
]

print("Running inference test on merged model...\n")
for i, raw_output in enumerate(test_cases, 1):
    inputs = verify_tokenizer(
        PROMPT_TEMPLATE.format(raw_output.strip()),
        return_tensors = "pt",
        truncation     = True,
        max_length     = 512
    ).to("cuda")

    with torch.no_grad():
        outputs = verify_model.generate(
            **inputs,
            max_new_tokens = 200,
            temperature    = 0.3,
            do_sample      = True,
            pad_token_id   = verify_tokenizer.eos_token_id
        )

    decoded   = verify_tokenizer.decode(outputs[0], skip_special_tokens=True)
    formatted = decoded.split("### Formatted Response:")[-1].strip()

    print(f"{'='*60}")
    print(f"Test {i} — Raw Input:")
    print(raw_output.strip())
    print(f"\nFormatted Output:")
    print(formatted)
    print()

print("✅ Merged model verified and working!")
print(f"   Saved at: {SAVE_PATH}")